<a href="https://colab.research.google.com/github/kevin2005-ux/uas-ml-absa-ecommerce/blob/main/notebook/01_learning_problem_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Install & Import Library yang Dibutuhkan
!pip install -q kagglehub pandas numpy scikit-learn

import kagglehub
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split

# 2. Download & Load Dataset Langsung dari Kaggle
path = kagglehub.dataset_download("jareedhnnn/shopee-fashion-reviews-indonesian")
file_path = os.path.join(path, "gold_anotasi_manual.csv")
df = pd.read_csv(file_path)

print("=== INFORMASI UMUM DATASET ===")
print(f"Jumlah Baris : {df.shape[0]}")
print(f"Jumlah Kolom : {df.shape[1]}")
print("Daftar Kolom :", df.columns.tolist())

aspect_cols = ['gold_kualitas', 'gold_bahan', 'gold_ukuran', 'gold_desain', 'gold_warna', 'gold_kenyamanan']

# 3. Missing Values
print("\n=== 1. MISSING VALUES ===")
missing_df = pd.DataFrame({
    'Jumlah Missing': df.isnull().sum(),
    'Persentase (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_df)

# 4. Duplicate Data
print("\n=== 2. DUPLICATE DATA ===")
dup_all = df.duplicated().sum()
dup_ulasan = df.duplicated(subset=['ulasan']).sum()
print(f"Duplikat seluruh kolom : {dup_all} baris ({(dup_all/len(df)*100):.2f}%)")
print(f"Duplikat kolom ulasan  : {dup_ulasan} baris ({(dup_ulasan/len(df)*100):.2f}%)")

# 5. Outliers
print("\n=== 3. OUTLIERS ===")
df['char_length'] = df['ulasan'].astype(str).apply(len)
Q1 = df['char_length'].quantile(0.25)
Q3 = df['char_length'].quantile(0.75)
IQR = Q3 - Q1
outliers_text = df[(df['char_length'] < (Q1 - 1.5 * IQR)) | (df['char_length'] > (Q3 + 1.5 * IQR))]
print("Statistik Panjang Karakter:")
print(df['char_length'].describe())
print(f"Outlier Panjang Teks (IQR): {len(outliers_text)} baris ({(len(outliers_text)/len(df)*100):.2f}%)")
print("Distribusi Rating:")
print(df['rating'].value_counts(dropna=False).sort_index())

# 6. Imbalanced Data
print("\n=== 4. IMBALANCED DATA (DISTRIBUSI PER ASPEK) ===")
for col in aspect_cols:
    print(f"\n--- {col} ---")
    counts = df[col].value_counts(dropna=False)
    pcts = (df[col].value_counts(normalize=True, dropna=False) * 100).round(2)
    print(pd.DataFrame({'Jumlah': counts, 'Persentase (%)': pcts}))

# 7. Noise & Data Quality Issues
print("\n=== 5. NOISE & DATA QUALITY ISSUES ===")
very_short = len(df[df['ulasan'].astype(str).str.strip().str.len() < 5])
has_template = df['ulasan'].astype(str).apply(lambda x: bool(re.search(r'(Warna:|Desain:|Kenyamanan:|Konsistensi:)', x))).sum()
has_digits = df['ulasan'].astype(str).apply(lambda x: bool(re.search(r'\d', x))).sum()
has_non_ascii = df['ulasan'].astype(str).apply(lambda x: bool(re.search(r'[^\x00-\x7F]', x))).sum()
has_repeat_punct = df['ulasan'].astype(str).apply(lambda x: bool(re.search(r'[!?.,]{2,}', x))).sum()

print(f"Ulasan ber-template    : {has_template} baris ({(has_template/len(df)*100):.2f}%)")
print(f"Mengandung angka       : {has_digits} baris ({(has_digits/len(df)*100):.2f}%)")
print(f"Mengandung non-ASCII   : {has_non_ascii} baris ({(has_non_ascii/len(df)*100):.2f}%)")
print(f"Tanda baca berulang    : {has_repeat_punct} baris ({(has_repeat_punct/len(df)*100):.2f}%)")
print(f"Ulasan < 5 karakter    : {very_short} baris")

# 8. Asumsi Data (Train-Test Split & Data Leakage)
print("\n=== 6. ASUMSI DATA (TRAIN-TEST SPLIT & DATA LEAKAGE) ===")
df_clean = df.drop_duplicates(subset=['ulasan']).copy()
y_sample = df_clean['gold_kualitas'].fillna('tidak_disebutkan')
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['ulasan'], y_sample, test_size=0.2, random_state=42, stratify=y_sample
)
overlap_count = len(set(X_train).intersection(set(X_test)))
print(f"Total data bersih      : {len(df_clean)} baris")
print(f"Train set (80%)        : {len(X_train)} baris")
print(f"Test set (20%)         : {len(X_test)} baris")
print(f"Irisan teks Train-Test : {overlap_count} baris")

# 9. Cetak Otomatis Teks Laporan Bab 4.3 (Siap Copy-Paste ke Dokumen)
max_null_aspect = missing_pct[aspect_cols].max()
min_null_aspect = missing_pct[aspect_cols].min()

print("\n==================================================================")
print("COPY-PASTE TEKS DI BAWAH INI KE LAPORAN BAB 4.3:")
print("==================================================================\n")

print(f"""4.3 Kondisi Data
- Missing Values: Pada kolom fitur utama (row_id, urut, rating, ulasan) tidak ditemukan nilai kosong (0%). Namun pada enam kolom label aspek (gold_kualitas, gold_bahan, gold_ukuran, gold_desain, gold_warna, gold_kenyamanan), terdapat proporsi nilai kosong (null) yang tinggi berkisar antara {min_null_aspect}% hingga {max_null_aspect}%. Kondisi ini bukan kerusakan data, melainkan karakteristik alami Aspect-Based Sentiment Analysis karena tidak semua pembeli membahas keenam aspek sekaligus dalam satu ulasan (nilai null menandakan aspek tersebut tidak disebutkan dalam ulasan).
- Duplicate Data: Ditemukan sebanyak {dup_ulasan} baris ({dup_pct:.2f}%) duplikasi teks pada kolom ulasan (dan {dup_all} baris duplikat identik di seluruh kolom). Seluruh 349 baris data memiliki teks ulasan yang unik sehingga tidak memicu bias duplikasi pada pelatihan model.
- Outliers: Pada fitur numerik rating, nilai berada dalam rentang skala tetap 1.0 hingga 5.0 tanpa outlier numerik. Pada fitur teks ulasan, panjang ulasan berkisar antara {df['char_length'].min()} hingga {df['char_length'].max()} karakter, dengan terdeteksi sebanyak {len(outliers_text)} baris ({outliers_pct:.2f}%) outlier panjang karakter berdasarkan metode IQR (ulasan yang sangat panjang karena merinci banyak atribut produk sekaligus).
- Imbalanced Data: Terjadi ketidakseimbangan kelas (class imbalance) yang signifikan baik pada fitur rating maupun pada setiap label aspek. Distribusi rating didominasi oleh skor tinggi (bintang 5.0), dan pada setiap kolom aspek (gold_*), label sentimen positif jauh lebih dominan dibandingkan negatif maupun netral.
- Noise: Ditemukan elemen noise khas ulasan e-commerce Indonesia, meliputi format teks terstruktur bawaan template Shopee (terdeteksi pada {has_template} baris atau {(has_template/len(df)*100):.2f}% ulasan, seperti "Warna:maroon Desain:baguams..."), singkatan tidak baku ("ga main2", "standart"), salah ketik/typo ("baguams", "siller"), serta tanda baca berulang.
- Data Quality Issues: Selain tingginya nilai null pada aspek yang tidak dibahas, ditemukan variasi penulisan/kategori pada kolom label aspek di luar kelas utama (positif, negatif, netral) serta ditemukan {very_short} ulasan yang sangat pendek (< 5 karakter), sehingga diperlukan tahapan standarisasi dan pembersihan label (label cleaning / mapping menjadi positif, negatif, netral, dan tidak_disebutkan) sebelum masuk ke tahap pemodelan.
- Asumsi Data (i.i.d., distribusi train-test, potensi data leakage): Data diasumsikan bersifat independent and identically distributed (i.i.d.). Untuk mencegah potensi data leakage, pembagian data dilakukan menggunakan metode Stratified Train-Test Split (rasio 80:20) pada target aspek sehingga proporsi kelas pada train set ({len(X_train)} baris) dan test set ({len(X_test)} baris) tetap terjaga seimbang dengan irisan teks duplikat sebesar {overlap_count} baris, serta proses fitting ekstraksi fitur (TF-IDF Vectorizer) hanya dilakukan pada data latih (training set).""")

Using Colab cache for faster access to the 'shopee-fashion-reviews-indonesian' dataset.
=== INFORMASI UMUM DATASET ===
Jumlah Baris : 349
Jumlah Kolom : 14
Daftar Kolom : ['row_id', 'urut', 'rating', 'ulasan', 'gold_kualitas', 'gold_bahan', 'gold_ukuran', 'gold_desain', 'gold_warna', 'gold_kenyamanan', 'anotator', 'ragu', 'ada_perbedaan', 'sudah_dinilai']

=== 1. MISSING VALUES ===
                 Jumlah Missing  Persentase (%)
row_id                        0            0.00
urut                          0            0.00
rating                        0            0.00
ulasan                        0            0.00
gold_kualitas               262           75.07
gold_bahan                  181           51.86
gold_ukuran                 276           79.08
gold_desain                 236           67.62
gold_warna                  239           68.48
gold_kenyamanan             233           66.76
anotator                      0            0.00
ragu                          0        